# ಪಾಠ 04 - ಸಾಧನ ಬಳಕೆ ವಿನ್ಯಾಸ ಮಾದರಿ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework (Python) ಬಳಸಿ AI ಏಜೆಂಟುಗಳಿಗೆ **ಸಾಧನ ಬಳಕೆ** ವಿನ್ಯಾಸ ಮಾದರಿಯನ್ನು ಕಲಿಯುತ್ತೀರಿ. ನಾವು ಒಳಗೊಂಡಿರುತ್ತೇವೆ:

- `@tool` ಡೆಕೋರೇಟರ್ ಮತ್ತು ಟೈಪ್ ಮಾಡಲಾದ ಪ್ಯಾರಾಮೀಟರ್ಸ್ ಬಳಸಿ ಫಂಕ್ಷನ್ ಸಾಧನಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು
- ಪ್ರತಿಯೊಂದು ಸಾಧನವೇ ಏನು ಮಾಡುತ್ತದೆ ಎಂದು ಮಾದರಿಗೆ ತಿಳಿಸುವುದಕ್ಕಾಗಿ ಸಾಧನ ಯೋಜನೆಯನ್ನು ಒದಗಿಸುವುದು
- `approval_mode` ಮೂಲಕ ಸಾಧನ ಕಾರ್ಯಾಚರಣೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು
- Pydantic ಮಾದರಿಗಳ ಮತ್ತು `response_format` ಮೂಲಕ **ಸಂರಚಿತ ouput** ಅನ್ನು ಹಿಂತಿರುಗಿಸುವುದು

ಈ ದೃಶ್ಯಾವಳಿ ಒಂದು **ಪ್ರವಾಸ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟು** ಆಗಿದ್ದು, ಇದು ದೃಶ್ಯಾವಳಿಗಳನ್ನು ಹುಡುಕುವುದು, ಲಭ್ಯತೆಯನ್ನು ಪರಿಶೀಲಿಸುವುದು, ಮತ್ತು ವಿಮಾನ ವಿಮರ್ಶನ ಮಾಹಿತಿ ಪಡೆಯುವುದು.


## ಸೆಟಪ್


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ಅಲಂಕಾರಕದೊಂದಿಗೆ ಉಪಕರಣಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು

`@tool` ಅಲಂಕಾರಕವು ಸರಳ ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಅನ್ನು ಏಜಂಟ್ ಕರೆಸಬಹುದಾದ ಉಪಕರಣವಾಗಿಸುತ್ತದೆ.
ಮುಖ್ಯ ಅಂಶಗಳು:

- **ಡಾಕ್ಸ್ಟ್ರಿಂಗ್** ಮಾದರಿ ನೋಡುವ ಉಪಕರಣ ವರ್ಣನೆಯಾಗಿ ಪರಿಗಣಿಸಲಾಗಿದೆ.
- **ಪ್ರಕಾರ ಸೂಚಿಕೆಗಳು** (`Annotated` ಸಹಿತ ವಿವರಣೆಗಳೊಂದಿಗೆ) ಉಪಕರಣ ಸ್ಕೀಮೆಯನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತವೆ.
- `approval_mode` ಬಳಕೆದಾರನು ಪ್ರತೀ ಕರೆ ಕಾರ್ಯಗತಗೊಳಿಸುವ ಹಿಂದೆ ಅನುಮೋದಿಸುವ ಅಗತ್ಯವಿದೆಯೇ ಎಂಬುದನ್ನು ನಿಯಂತ್ರಿಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ಹಲವಾರು ಸಾಧನಗಳೊಂದಿಗೆ ಏಜೆಂಟನ್ನು ರಚಿಸುವುದು

ಮಾಡೆಲ್ ಬಳಕೆದಾರರ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸಲು ಅಗತ್ಯವಿರುವ ಯಾವುದೇ ಸಾಧನಗಳನ್ನು ಕರೆಸಿಕೊಳ್ಳಲು ಕ್ಲೈಂಟ್‌ಗೆ ಎಲ್ಲಾ ಮೂರು ಸಾಧನಗಳನ್ನು ಒದಗಿಸಿ.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ಸಾಧನಗಳೊಂದಿಗೆ ಸಂರಚಿತ ಔಟ್‌ಪುಟ್

`response_format` ಅನ್ನು ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಯಾಗಿ ಸಜ್ಜುಗೊಳಿಸುವ ಮೂಲಕ, ಏಜೆಂಟ್ ಸ್ವಾತಂತ್ರ್ಯಪೂರ್ಣ ಪಠ್ಯದ ಬದಲಾಗಿ ಚೆನ್ನಾಗಿ ಟೈಪ್ ಮಾಡಲಾದ JSON ವಸ್ತುವನ್ನು ಹಿಂತಿರುಗಿಸಲು ಬಾಧ್ಯವಾಗುತ್ತದೆ. ನಂತರದ ಕೋಡ್ ಫಲಿತಾಂಶವನ್ನು ಪ್ರೋಗ್ರಾಮ್ಯಾಟಿಕಲ್ ಆಗಿ ಬಳಸಬೇಕಾದಾಗ ಇದು ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ಪಾಠ್ಯಾನುಮೋದನೆ ಮಾದರಿಗಳು

`@tool` ನಲ್ಲಿ `approval_mode` ಪರಿಮಿತಿ ಟೂಲ್ ಕರೆಗಳು ಕಾರ್ಯನಿರ್ವಹಿಸುವ ಮೊದಲು ಮಾನವ ಅನುಮೋದನೆ ಬೇಕಾದುದನ್ನು ನಿಯಂತ್ರಿಸುತ್ತದೆ:

| ಮಾದರಿ | ನಡೆ|
|---|---|
| `"never_require"` | ಟೂಲ್ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಕಾರ್ಯಗತಗೊಳ್ಳುತ್ತದೆ — ಬಳಕೆದಾರ ದೃಢೀಕರಣ ಅಗತ್ಯವಿಲ್ಲ. |
| `"always_require"` | ಪ್ರತಿ ಕರೆ ಕಾರ್ಯಗತಗೊಳ್ಳುವುದಕ್ಕೆ ಮೊದಲು ಬಳಕೆದಾರರಿಂದ ಅನುಮೋದನೆ ಪಡೆಯಬೇಕು. |

ಸರ್ಕಾರಿ ಪರಿಣಾಮವಿರುವ ಟೂಲ್‌ಗಳಿಗಾಗಿ (ಉದಾ: ವಿಮಾನ ಟಿಕೆಟ್ ಬುಕಿಂಗ್, ಕ್ರೆಡಿಟ್ ಕಾರ್ಡ್ ದಂಡಿಸುವಿಕೆ) `"always_require"` ಬಳಸಿ, ಹೀಗಾಗಿ ಮಾನವನು ಪ್ರಕ್ರિયામાં ಭಾಗಿಯಾಗಿರುತ್ತಾನೆ.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಕಲಿತಿರಿ:

1. ಟೂಲ್ಗಳನ್ನು `@tool` ಡೆಕ್ಯಾರೆಟರ್ ಬಳಸಿ ಟೈಪ್‌ಡ್ ಪ್ಯಾರಾಮೀಟರ್‌ಗಳು ಮತ್ತು ಟೂಲ್ಗೆ ಸ್ಕೆಮಾ ಸೇವೆ ನೀಡುವ ಡಾಕ್‌ಸ್ಟ್ರಿಂಗ್‌ಗಳೊಂದಿಗೆ ವ್ಯಾಖ್ಯಾನಿಸುವುದು.
2. ಜಟಿಲ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸಲು ಏಜೆಂಟ್ ಸರಣಿಯಲ್ಲಿ ಕರೆಯಬಹುದಾದ ಅನೇಕ ಟೂಲ್ಗಳನ್ನು ಸಮನ್ವಯಿಸುವುದು.
3. `response_format` ಆಗಿ ಪೈಡೆಂಟಿಕ್ ಮಾದರಿಯನ್ನು ಪಾಸಿನ್ನು ರಚನೆಗೊಳಿಸಿದ ಔಟ್ಪುಟ್ ಅನ್ನು ಹಿಂತಿರುಗಿಸುವುದು.
4. ಸೂಕ್ಷ್ಮ ಕಾರ್ಯಗಳಿಗೆ ಮಾನವನು ನಿಯಂತ್ರಣದಲ್ಲಿ ಇರಲು `approval_mode`ೊಂದಿಗೆ ಟೂಲ್ ಅನುಮೋದನೆ ನಿಯಂತ್ರಿಸುವುದು.

ಈ ಮಾದರಿಗಳು ವಿಶ್ವಾಸಾರ್ಹ, ಉತ್ಪಾದನಾ-ತಯಾರ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವ ಮೂಲಾಧಾರವಾಗಿದೆ, ಅವು ಹೊರಗಿನ ವ್ಯವಸ್ಥೆಗಳೊಂದಿಗೆ ಸುರಕ್ಷಿತವಾಗಿ ಸಂವಹನ ಮಾಡಬಹುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ನಿರಾಕರಣೆ**:  
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯತ್ತ ಪ್ರಯತ್ನಿಸಿದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳಿರಬಹುದು ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿಟ್ಟುಕೊಳ್ಳಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ದಸ್ತಾವೇಜಿನ ತತ್ವಜ್ಞಾನಿಕ ನಿಗಾದಿ ಮೂಲವಾದರೆ. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅಂಗೀಕೃತ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಅನರ್ಥಗಳು ಅಥವಾ ತಪ್ಪು ಅರ್ಥಾಂಶಗಳಿಗೆ ನಾವು ಹೊಣೆಗಾರರಾಗಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
